# Ligand preparation

Code to prepare ligands of interest for the RASSCoL pipeline.

## Directory creation

In [2]:
# builtins
from pathlib import Path
import subprocess

# local
from rasscol_src.general_utils import *
from rasscol_src.rasscol_utils import *

## USER INPUT ##
data_dir = Path('./example/CCHept_DPH')
ligand_name = 'DPH'
ligand_short_name = 'DPH'
smiles = 'C1=CC=C(C=C1)/C=C/C=C/C=C/C2=CC=CC=C2'
molecular_formula = 'C18H16'
number_aromatic_rings = 2
number_non_aromatic_rings = 0
cleanup_speed = 'fast'
#################

## Ligand PDBQT creation

In [3]:
# usually takes several minutes

data_dir.mkdir(parents=True, exist_ok=True)

ligand_smi_path = data_dir / f'{ligand_name}.smi'
ligand_pdbqt_path = ligand_smi_path.with_suffix('.pdbqt')
ligand_pdb_path = ligand_smi_path.with_suffix('.pdb')

if not ligand_pdbqt_path.exists():
        
    with ligand_smi_path.open('w') as f:
        f.write(smiles)

    # obabel will try to minimise the 3d structure. This option takes time. 
    # there are different speeds (number of cycles) you can chose from.
    #
    # --------------------------------------------------------------------------------------------
    # option	    description
    # --------------------------------------------------------------------------------------------
    # fastest	    No cleanup
    # fast	        Force field cleanup (100 cycles)
    # med (default)	Force field cleanup (100 cycles) + Fast rotor search (only one permutation)
    # slow	        Force field cleanup (250 cycles) + Fast rotor search (permute central rotors)
    # slowest	    Force field cleanup (500 cycles) + Slow rotor search
    # better	    Same as slow
    # best	        Same as slowest
    # dist, dg	    Use distance geometry method (unstable)

    obabel_path = Path('/usr/bin/obabel')
    obabel_log_path = data_dir / 'obabel.log'

    obabel_cmd = [
        obabel_path, 
        '-ismi', ligand_smi_path,
        '-opdbqt', '-O', ligand_pdbqt_path,
        '--gen3d', cleanup_speed,
        '-p', '7.4'
    ]

    with obabel_log_path.open('w') as f:
        subprocess.run(obabel_cmd, stdout=f, stderr=f)
        
    tidy_ligand_pdbqt(ligand_pdbqt_path, ligand_name, ligand_short_name)

ligand_coords = get_pdbqt_coords(ligand_pdbqt_path)
mol_length = get_mol_len(ligand_coords)
bondi_vol = calc_bondi_vol(molecular_formula, number_non_aromatic_rings, number_aromatic_rings)

print(f"Bondi volume: {bondi_vol} Å3")
print(f"Length: {mol_length:.1f} Å")


Bondi volume: 269.12 Å3
Length: 14.1 Å
